# Great Expectations - Валидация музыкальных данных
Задание: Проверяем данные с помощью Great Expectations

In [1]:
import pandas as pd
import great_expectations as gx
import json
from datetime import datetime

In [2]:
df = pd.read_csv('dataset1.csv')
print(f'Размер данных: {df.shape}')

Размер данных: (114000, 21)


In [3]:
# Удаляем лишние колонки
columns_to_drop = [col for col in df.columns if col.startswith('Unnamed') or col == 'index']
if columns_to_drop:
    df = df.drop(columns=columns_to_drop)
print(f'Размер после очистки: {df.shape}')

Размер после очистки: (114000, 20)


In [4]:
# Получаем уникальные жанры
UNIQUE_GENRES = set(df['track_genre'].dropna().unique())
n_genres = len(UNIQUE_GENRES)
print(f'Уникальных жанров: {n_genres}')

Уникальных жанров: 114


In [5]:
# Создаём file context
import shutil
import os

# Очищаем старый context
context_dir = 'great_expectations'
if os.path.exists(context_dir):
    shutil.rmtree(context_dir)

context = gx.get_context(mode='file', context_root_dir=context_dir)
datasource = context.data_sources.add_pandas(name='music_data_source')
asset = datasource.add_dataframe_asset(name='music_data_asset')
batch_request = asset.build_batch_request(options={'dataframe': df})
print('Context создан')

Context создан


In [6]:
# Создаём Expectation Suite
expectation_suite = gx.ExpectationSuite(name='music_data_expectations')
expectation_suite = context.suites.add(expectation_suite)
print(f'Suite создан: {expectation_suite.name}')

Suite создан: music_data_expectations


### Добавление ожиданий в Suite

In [7]:
# Проверка обязательных колонок
required_columns = [
    'track_id', 'artists', 'album_name', 'track_name', 'popularity',
    'duration_ms', 'explicit', 'danceability', 'energy', 'key',
    'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness',
    'liveness', 'valence', 'tempo', 'time_signature', 'track_genre'
]

for col in required_columns:
    expectation_suite.add_expectation(
        gx.expectations.ExpectColumnToExist(column=col)
    )
print(f'Добавлено проверок на наличие колонок: {len(required_columns)}')

Добавлено проверок на наличие колонок: 20


In [8]:
# Проверка на отсутствие пропусков
for col in required_columns:
    expectation_suite.add_expectation(
        gx.expectations.ExpectColumnValuesToNotBeNull(column=col)
    )
print(f'Добавлено проверок на пропуски: {len(required_columns)}')

Добавлено проверок на пропуски: 20


In [9]:
# Проверка track_id: длина строки строго 22 символа
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='track_id', type_='str')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValueLengthsToEqual(column='track_id', value=22)
)

ExpectColumnValueLengthsToEqual(id='98adcc6d-21fa-412b-9c7a-1c8c488319f6', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='track_id', mostly=1, row_condition=None, condition_parser=None, value=22.0)

In [10]:
# Проверка artists: длина строки от 2 до 512 символов
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='artists', type_='str')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValueLengthsToBeBetween(
        column='artists', min_value=2, max_value=512, strict_min=True, strict_max=True
    )
)

ExpectColumnValueLengthsToBeBetween(id='967b5014-ca62-43fa-9eb2-86f5519c4c07', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='artists', mostly=1, row_condition=None, condition_parser=None, min_value=2, max_value=512, strict_min=True, strict_max=True)

In [11]:
# Проверка album_name: длина строки от 2 до 512 символов
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='album_name', type_='str')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValueLengthsToBeBetween(
        column='album_name', min_value=2, max_value=512, strict_min=True, strict_max=True
    )
)

ExpectColumnValueLengthsToBeBetween(id='51f680b8-dd7f-42e5-91d4-c9e15bf961e4', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='album_name', mostly=1, row_condition=None, condition_parser=None, min_value=2, max_value=512, strict_min=True, strict_max=True)

In [12]:
# Проверка track_name: длина строки от 2 до 512 символов
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='track_name', type_='str')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValueLengthsToBeBetween(
        column='track_name', min_value=2, max_value=512, strict_min=True, strict_max=True
    )
)

ExpectColumnValueLengthsToBeBetween(id='0fb53cf4-043c-470d-a279-58581aa3d60e', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='track_name', mostly=1, row_condition=None, condition_parser=None, min_value=2, max_value=512, strict_min=True, strict_max=True)

In [13]:
# Проверка popularity: int от 0 до 100
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='popularity', type_='int')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='popularity', min_value=0, max_value=100, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='948a76dd-0fe1-4f2a-8284-b770298a0f5e', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='popularity', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=100.0, strict_min=False, strict_max=False)

In [14]:
# Проверка duration_ms: int от 0 (не включительно) до 5237760 (включительно)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='duration_ms', type_='int')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='duration_ms', min_value=0, max_value=5237760, strict_min=True, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='41eead91-5d4d-4846-95dd-2424c1db0084', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='duration_ms', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=5237760.0, strict_min=True, strict_max=False)

In [15]:
# Проверка explicit: bool
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='explicit', type_='bool')
)

ExpectColumnValuesToBeOfType(id='d2e90cb2-d0ad-4005-9875-d23e2fabbaf6', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='explicit', mostly=1, row_condition=None, condition_parser=None, type_='bool')

In [16]:
# Проверка danceability: float от 0 до 1
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='danceability', type_='float')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='danceability', min_value=0, max_value=1, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='06b3f3c3-12e1-486a-9327-97629002a887', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='danceability', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=1.0, strict_min=False, strict_max=False)

In [17]:
# Проверка energy: float от 0 до 1
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='energy', type_='float')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='energy', min_value=0, max_value=1, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='dc5d6bfa-fae9-4a0c-a50c-459e19b47076', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='energy', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=1.0, strict_min=False, strict_max=False)

In [18]:
# Проверка key: int от 0 до 11
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='key', type_='int')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='key', min_value=0, max_value=11, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='0971bee4-ec58-4b97-b490-02b85e2bb251', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='key', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=11.0, strict_min=False, strict_max=False)

In [19]:
# Проверка loudness: float от -45 до 5
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='loudness', type_='float')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='loudness', min_value=-45, max_value=5, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='31c09a80-e8d9-4143-9419-2f59102f71da', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='loudness', mostly=1, row_condition=None, condition_parser=None, min_value=-45.0, max_value=5.0, strict_min=False, strict_max=False)

In [20]:
# Проверка mode: float от 0 до 1
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='mode', type_='float')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='mode', min_value=0, max_value=1, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='f5d85dc8-12da-4aeb-bf79-a17667e247e7', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='mode', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=1.0, strict_min=False, strict_max=False)

In [21]:
# Проверка speechiness: float от 0 до 1
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='speechiness', type_='float')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='speechiness', min_value=0, max_value=1, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='267071b8-03dc-41f9-ba2f-40bf7b11b123', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='speechiness', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=1.0, strict_min=False, strict_max=False)

In [22]:
# Проверка acousticness: float от 0 до 1
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='acousticness', type_='float')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='acousticness', min_value=0, max_value=1, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='cf962bea-a3f8-4900-94e5-c3b40d554ca9', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='acousticness', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=1.0, strict_min=False, strict_max=False)

In [23]:
# Проверка instrumentalness: float от 0 до 1
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='instrumentalness', type_='float')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='instrumentalness', min_value=0, max_value=1, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='2caed7c8-d801-45c3-b4e6-9c426b78b070', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='instrumentalness', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=1.0, strict_min=False, strict_max=False)

In [24]:
# Проверка liveness: float от 0 до 1
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='liveness', type_='float')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='liveness', min_value=0, max_value=1, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='6e82f619-bc9d-4a97-9065-212776cb8c68', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='liveness', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=1.0, strict_min=False, strict_max=False)

In [25]:
# Проверка valence: float от 0 до 1
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='valence', type_='float')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='valence', min_value=0, max_value=1, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='3cc90074-ecc2-4b0c-a124-ba8e600f34f7', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='valence', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=1.0, strict_min=False, strict_max=False)

In [26]:
# Проверка tempo: float от 0 до 256
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='tempo', type_='float')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='tempo', min_value=0, max_value=256, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='d4b689e9-b47d-4638-8c1f-bced9b62f49f', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='tempo', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=256.0, strict_min=False, strict_max=False)

In [27]:
# Проверка time_signature: int от 0 до 5
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='time_signature', type_='int')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='time_signature', min_value=0, max_value=5, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='4fd7865f-3365-46cd-b339-8ff21399798e', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='time_signature', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=5.0, strict_min=False, strict_max=False)

In [28]:
# Проверка track_genre: str, ограничение на 114 уникальных жанров
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='track_genre', type_='str')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeInSet(column='track_genre', value_set=list(UNIQUE_GENRES))
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnUniqueValueCountToBeBetween(
        column='track_genre', min_value=n_genres, max_value=n_genres, strict_min=False, strict_max=False
    )
)
print(f'Добавлена проверка track_genre с {n_genres} уникальными значениями')

Добавлена проверка track_genre с 114 уникальными значениями


### Сохранение Expectation Suite в JSON

In [29]:
# Сохраняем suite в JSON файл
suite_json = expectation_suite.to_json_dict()
with open('music_data_expectations.json', 'w', encoding='utf-8') as f:
    json.dump(suite_json, f, indent=2, ensure_ascii=False)
print(f'Expectation Suite сохранён в music_data_expectations.json')
print(f'Количество ожиданий: {len(expectation_suite.expectations)}')

Expectation Suite сохранён в music_data_expectations.json
Количество ожиданий: 80


### Запуск проверки и получение результатов

In [ ]:
# Запускаем валидацию через validator
validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite=expectation_suite
)

validation_result = validator.validate()
print(f'Проверка завершена: {validation_result.success}')

# Сохраняем результат валидации в store контекста для Data Docs
from great_expectations.core.run_identifier import RunIdentifier
run_id = RunIdentifier(run_name="music_data_validation")
context.validation_results.add(key=run_id, value=validation_result)
print('Результаты сохранены в validation_results store')

### Сохранение результатов проверки в JSON

In [31]:
results_dict = validation_result.to_json_dict()
with open('music_data_validation_results.json', 'w', encoding='utf-8') as f:
    json.dump(results_dict, f, indent=2, ensure_ascii=False)
print('Результаты проверки сохранены в music_data_validation_results.json')

Результаты проверки сохранены в music_data_validation_results.json


### Генерация HTML-отчёта (Data Docs)

In [ ]:
# Очищаем старые Data Docs
data_docs_dir = os.path.join('great_expectations', 'uncommitted', 'data_docs')
if os.path.exists(data_docs_dir):
    shutil.rmtree(data_docs_dir)

# Строим Data Docs для валидаций
context.build_data_docs()
print('HTML-отчёт сгенерирован')

# Путь к отчёту
index_path = os.path.abspath(os.path.join(data_docs_dir, 'local_site', 'index.html'))
print(f'Путь к отчёту: {index_path}')

# Проверяем размер файла
import os
if os.path.exists(index_path):
    size = os.path.getsize(index_path)
    print(f'Размер файла: {size} байт')
    if size < 1000:
        print('ВНИМАНИЕ: Файл слишком маленький, возможно отчёт пустой!')
        # Читаем и выводим первые строки
        with open(index_path, 'r') as f:
            content = f.read()[:500]
            print(f'Содержимое: {content}')

In [ ]:
# Генерация HTML-отчёта из результатов валидации
import os

# Читаем результаты валидации
with open('music_data_validation_results.json', 'r', encoding='utf-8') as f:
    results = json.load(f)

# Генерируем HTML-отчёт
passed = sum(1 for r in results.get('results', []) if r.get('success'))
failed = len(results.get('results', [])) - passed
status = '✅ Все проверки пройдены' if results.get('success') else '❌ Обнаружены проблемы'

html = f'''<!DOCTYPE html>
<html lang="ru">
<head>
    <meta charset="UTF-8">
    <title>Отчёт валидации Great Expectations</title>
    <style>
        body {{ font-family: Arial, sans-serif; margin: 40px; background: #f5f5f5; }}
        .container {{ max-width: 1200px; margin: 0 auto; background: white; padding: 30px; border-radius: 8px; }}
        h1 {{ color: #333; border-bottom: 3px solid #4CAF50; padding-bottom: 10px; }}
        .summary {{ padding: 20px; border-radius: 8px; margin: 20px 0; }}
        .summary.success {{ background: #e8f5e9; border-left: 4px solid #4CAF50; }}
        .summary.failure {{ background: #ffebee; border-left: 4px solid #f44336; }}
        table {{ width: 100%; border-collapse: collapse; margin: 20px 0; }}
        th, td {{ padding: 12px; text-align: left; border-bottom: 1px solid #ddd; }}
        th {{ background: #f5f5f5; }}
        .status {{ padding: 4px 12px; border-radius: 4px; font-weight: bold; }}
        .status.pass {{ background: #c8e6c9; color: #2e7d32; }}
        .status.fail {{ background: #ffcdd2; color: #c62828; }}
    </style>
</head>
<body>
    <div class="container">
        <h1>📊 Отчёт валидации данных</h1>
        <div class="summary {'success' if results.get('success') else 'failure'}">
            <h2>Общая сводка</h2>
            <p><strong>Статус:</strong> {status}</p>
            <p><strong>Всего проверок:</strong> {len(results.get('results', []))}</p>
            <p><strong>Пройдено:</strong> {passed}</p>
            <p><strong>Провалено:</strong> {failed}</p>
        </div>
        <h2>Детали проверок</h2>
        <table>
            <thead><tr><th>Статус</th><th>Проверка</th><th>Колонка</th><th>Параметры</th></tr></thead>
            <tbody>'''

for result in results.get('results', []):
    config = result.get('expectation_config', {})
    kwargs = config.get('kwargs', {})
    exp_type = config.get('type', 'unknown')
    success = result.get('success', False)
    column = kwargs.get('column', 'N/A')
    exp_name = exp_type.replace('expect_', '').replace('_', ' ').title()
    params = []
    for k in ['min_value', 'max_value', 'value', 'type_', 'column']:
        if k in kwargs and k != 'column':
            params.append(f"{k}: {kwargs[k]}")
    params_str = ', '.join(params) if params else '-'
    status_class = 'pass' if success else 'fail'
    status_text = '✓' if success else '✗'
    html += f'''<tr><td><span class="status {status_class}">{status_text}</span></td><td>{exp_name}</td><td>{column}</td><td>{params_str}</td></tr>\n'''

html += '''</tbody></table></div></body></html>'''

# Сохраняем отчёт
os.makedirs('great_expectations/uncommitted/data_docs/local_site', exist_ok=True)
with open('great_expectations/uncommitted/data_docs/local_site/index.html', 'w', encoding='utf-8') as f:
    f.write(html)

print(f'HTML-отчёт сгенерирован')
print(f'Путь: {os.path.abspath("great_expectations/uncommitted/data_docs/local_site/index.html")}')
print(f'Размер: {os.path.getsize("great_expectations/uncommitted/data_docs/local_site/index.html")} байт')